# Image-GMM barycenter scaling
MW2 full-tuple LP versus halving min-LSOT. Select a GPU runtime; CPU also works.


In [ ]:
from pathlib import Path
import subprocess, sys, os
repo = Path("/content/LSOT_GMM")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/tmp0810/LSOT_GMM.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)


## Pilot or full sweep
Choose one mode. Changing settings requires a new output directory.


In [ ]:
PILOT = True
output = "results/barycenter_scaling_pilot" if PILOT else "results/barycenter_scaling"
command = [sys.executable, "-m", "experiments.barycenter.scaling",
           "--config", "experiments/barycenter/configs/scaling.json",
           "--device", "auto", "--output-dir", output]
if PILOT:
    command += ["--component-counts", "5", "10", "20", "30", "--seeds", "0"]
subprocess.run(command, check=True)


## Inspect statuses and quality
An intentional skip, a memory cap and a real OOM have different status labels.


In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv(Path(output) / "summary.csv"))
for name in ["total_ms.png", "relative_mw2_gap.png", "plans_time.png"]:
    path = Path(output) / name
    if path.exists():
        display(Image(filename=str(path)))


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(output + "_export", "zip", root_dir=output)
files.download(archive)
